## Inferencce and Evaluation

### 0 - Sagemaker config

In [ ]:
import boto3
import pandas as pd
import os
from io import StringIO

import sagemaker
from sagemaker.image_uris import retrieve
from sagemaker.serializers import CSVSerializer

In [ ]:
# Region identification
region = sagemaker.Session().boto_region_name
print("AWS Region: {}".format(region))

# Role identification
role = sagemaker.get_execution_role()
print("RoleArn: {}".format(role))

# SageMaker session
sagemaker_session = sagemaker.Session()

### 1 - Inference

**Load local data (real-time request)**

In [ ]:
data_dir_test = "./data/pd/test"
df_test = pd.read_csv(os.path.join(data_dir_test, 'test.csv'), header=None)

In [ ]:
# Separating variables and target
df_y_test = df_test[0]
df_x_test = df_test.loc[:, df_test.columns != 0]

In [ ]:
# min 1 max 1000
num_predicciones = 50
records_to_predict = df_x_test[:num_predicciones]

In [ ]:
# Serialize data 
# by default sagemaker expects a comma-separated csv
csv_file = StringIO()
records_to_predict.to_csv(csv_file, sep=",", header=False, index=False)
payload_as_csv = csv_file.getvalue()

**Inference**

In [ ]:
from sagemaker.predictor import Predictor

In [ ]:
# Instantiate a predictor object that we will use to send data to the endpoint and get back predictions
endpoint_name = "my-endpoint-serverless"
predictor = Predictor(endpoint_name=endpoint_name,
                      sagemaker_session=sagemaker.Session(),
                      serializer=CSVSerializer())

In [ ]:
# Send the data to the endpoint and get back predictions
predicc_res = predictor.predict(payload_as_csv).decode('utf-8')

### 2 - Model evaluation

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# The endpoint returns a string with the predicted values separated by commas, we need to split the string and convert the values to floats
vals = predicc_res.split(',')
predicciones = [round(float(num)) for num in vals]

In [ ]:
# Accuracy evaluation
accuracy_score(df_y_test[:num_predicciones], predicciones)

In [ ]:
# Confusion matrix evaluation
cm = confusion_matrix(df_y_test[:num_predicciones], predicciones)

In [ ]:
# Plotting the confusion matrix
sns.heatmap(cm,
            annot=True,
            fmt='g',
            xticklabels=['Neg','Pos'],
            yticklabels=['Neg','Pos'])
plt.ylabel('Prediction',fontsize=13)
plt.xlabel('Actual',fontsize=13)
plt.title('Confusion Matrix',fontsize=17)
plt.show()

### 3 - Delete endpoint and models

In [ ]:
# sdk de aws
sagemaker_bt3 = boto3.client("sagemaker", region_name=region)

**Delete endpoints**

In [ ]:
endpoint_list = sagemaker_bt3.list_endpoints()
endpoints = [endpoint['EndpointName'] for endpoint in endpoint_list['Endpoints']]
print(f"Endpoints to delete: {endpoints}")

In [ ]:
for endpoint in endpoints:
    print(f"Deleting endpoint: {endpoint}")
    sagemaker_bt3.delete_endpoint(EndpointName=endpoint)

**Delete endpoint's configuration**

In [ ]:
endpoints_config_res = sagemaker_bt3.list_endpoint_configs()
endpoints_config = [endpoint['EndpointConfigName'] for endpoint in endpoints_config_res['EndpointConfigs']]
print(f"Endpoint configs to delete: {endpoints_config}")

In [ ]:
for endpoint_conf in endpoints_config:
    print(f"Deleting endpoint config: {endpoint_conf}")
    sagemaker_bt3.delete_endpoint_config(EndpointConfigName=endpoint_conf)

**Delete models**

In [ ]:
models_list = sagemaker_bt3.list_models()
models = [model['ModelName'] for model in models_list['Models']]
print(f"Models to delete: {models}")

In [ ]:
for model in models:
    print(f"Deleting model: {model}")
    sagemaker_bt3.delete_model(ModelName=model)